# py2Dmol in Colab — the things Colab breaks

Colab renders **every cell output in its own iframe**. That one fact decides the
whole live path: in Jupyter a `v.add()` in a later cell can reach into the viewer
directly, because everything is one document. Here it cannot, and
`BroadcastChannel` is the only bridge.

`tests/colab.py` reproduces that shape locally and runs on every commit. This
notebook is the other half — the part a local harness cannot answer, because
Colab's frontend is closed: **does it work in the real thing.**

**Run all cells top to bottom, one at a time**, and read the *expect* line under
each. The last two sections check themselves and print PASS/FAIL.

| # | what it covers |
|---|---|
| 1 | a later cell reaching the viewer at all |
| 2 | colour, SSE, contacts and a frame's own colour, set *after* the frames |
| 3 | taking every one of them off again |
| 4 | reopening the notebook with no runtime and no network |
| 5 | self-checks: the emitted payload, and the channel itself |


## 0 · Install

Drag the wheel into the Files pane on the left first, or use the git line if the
branch is pushed.


In [ ]:
%pip install -q /content/py2dmol-1.7.0-py3-none-any.whl
# ...or, once the branch is pushed:
# %pip install -q 'git+https://github.com/YOUR-USER/py2Dmol@main'

import py2Dmol, numpy as np
print('py2Dmol', getattr(py2Dmol, '__version__', 'no __version__ attribute'))


## 1 · A later cell reaching the viewer

`show()` first, with nothing in it — the *live* mode. Everything after this
arrives over the channel.

**Expect:** an empty viewer panel.


In [ ]:
def helix(dz=0.0, n=60):
    t = np.linspace(0, 8 * np.pi, n)
    return np.stack([np.cos(t) * 5, np.sin(t) * 5, t * 1.5 + dz], axis=1)

v = py2Dmol.view(style='richardson')
v.show()


Now three frames, **one per cell** — three separate output iframes, which is
the whole point.

**Expect:** the helix appears, and after the third cell the frame player above
the canvas reads **3** frames.


In [ ]:
v.add(helix(0), align=False, name='m')


In [ ]:
v.add(helix(4), align=False, name='m')


In [ ]:
v.add(helix(8), align=False, name='m')


> **If the viewer above is still empty**, that is the bug this notebook exists
> for and it means `BroadcastChannel` is not crossing Colab's iframes. Skip to
> section 5, which reports on the channel directly.


## 2 · Metadata set after the frames

These travel as *changed metadata* rather than as frames — a second path, which
until recently had a second, drifted copy of the code that applies it.


**Expect:** the whole helix turns red.


In [ ]:
v.set_color('red', name='m')


**Expect:** residues 10–29 are forced to helix — visibly a fatter ribbon than
the automatic assignment around them. *(This did nothing at all before: the
incremental applier had no `sse` branch.)*


In [ ]:
v.set_sse('H', name='m', position=(10, 30))


**Expect:** a dashed contact line springs between residues 5 and 40.


In [ ]:
v.add_contacts([[5, 40, 1.0]], name='m')


**Expect:** step the player to **frame 2 of 3** — it is blue, and the other two
are still red. *(Also did nothing before: a frame is delivered once, so a colour
set on a frame the viewer already holds had no route to it.)*


In [ ]:
v.set_color('blue', name='m', frame=1)


## 3 · Taking it all off again

Setting and unsetting were not the same path. A field that went away simply
stopped appearing in the update — never unequal to anything, never sent — so it
stayed on screen for the life of the session.


**Expect:** frame 2 goes back to red with the others.


In [ ]:
v.set_color(None, name='m', frame=1)


**Expect:** the contact line disappears.


In [ ]:
v.add_contacts([], name='m')


**Expect:** residues 10–29 return to the automatic assignment.


In [ ]:
v.set_sse(None, name='m', position=(10, 30))


**Expect:** the red goes and the default colouring comes back.


In [ ]:
v.set_color(None, name='m')


**Expect:** chain colouring, then only chain B keeps it.

*(A selective clear must take what it names and nothing else.)*


In [ ]:
v.add(helix(0), align=False, name='two', chains=['A'] * 30 + ['B'] * 30)
v.set_color('orange', name='two', chain='A')
v.set_color('purple', name='two', chain='B')


In [ ]:
v.set_color(None, name='two', chain='A')   # B stays purple


## 4 · Reopening with no runtime and no network

**This is the one that was broken, and it is a manual step.**

`BroadcastChannel` does not retain. On a reopen every output iframe loads at
once, and the viewer's is half a megabyte against an update cell's kilobyte — so
the small ones routinely post before the viewer's channel exists, and their
frames were lost for good. The viewer now announces itself with `viewerReady`
and the update cells answer; because they answer in whatever order the iframes
happen to run, and `seq` is a watermark that discards anything below its high
mark, the viewer holds the replay for 800 ms and applies it **sorted**.

Do this:

1. **File → Save**.
2. **Runtime → Disconnect and delete runtime**.
3. Reload the browser tab (or close it and open the notebook again).
4. **Do not run anything.** Scroll to the viewer in section 1.

**Expect:** the viewer is there, drawing the helix, with **3 frames** in the
player — rebuilt entirely from saved cell outputs, with no kernel and nothing
fetched. Turning off your network first makes the point sharper.

> Note `persistence=False` cannot do this and is not meant to: it is one mailbox
> cell, overwritten, holding only the last unsent delta, so a reopen replays one
> frame. `persistence=True` — the default — writes a cell per `add()`.


## 5 · Self-checks

### 5a · What Python emits

No browser involved: this drives a second viewer with `display` captured, and
reads the payloads it produced.


In [ ]:
import json, re, types, sys
from IPython.display import HTML as _RealHTML
import py2Dmol.viewer as _vm

CELLS = []
class _Handle:
    def __init__(self, i): self.i = i
    def update(self, o): CELLS[self.i]['html'] = o._html
class _HTML:
    def __init__(self, s): self._html = s
def _display(o, display_id=None):
    CELLS.append({'id': display_id, 'html': getattr(o, '_html', str(o))})
    return _Handle(len(CELLS) - 1)

_real_display, _real_html = _vm.display, _vm.HTML
_vm.display, _vm.HTML = _display, _HTML
try:
    w = py2Dmol.view()
    w.show()
    for k in range(3):
        w.add(helix(4 * k), align=False, name='m')
    w.set_color('red', name='m')
    w.set_color('blue', name='m', frame=1)
    w.add_contacts([[5, 40, 1.0]], name='m')
    w.set_color(None, name='m')          # a REMOVAL
    w.add_contacts([], name='m')         # another
finally:
    _vm.display, _vm.HTML = _real_display, _real_html

viewer_html = CELLS[0]['html']
updates = [c['html'] for c in CELLS[1:]]
payloads = []
for h in updates:
    m = re.search(r'const p=(\{.*?\});const', h, re.S)
    if m:
        payloads.append(json.loads(m.group(1)))

bad = []
def check(ok, msg):
    print(('PASS  ' if ok else 'FAIL  ') + msg)
    if not ok: bad.append(msg)

check(len(CELLS) - 1 >= 7,
      f'persistence=True writes one cell per call ({len(CELLS) - 1} update cells)')
check(all('viewerReady' in h for h in updates),
      'every update cell answers viewerReady - without this a reopen loses them')
check(not re.search(r'(?:src|href)\s*=\s*["\'](?:https?:)?//', viewer_html),
      'the viewer payload names no external resource (reopens offline)')
check('fetch(' not in viewer_html,
      'the viewer payload fetches nothing')
metas = [p.get('meta', {}).get('m', {}) for p in payloads]
check(any('frame_colors' in m for m in metas),
      "a frame's own colour travels as frame_colors")
check(any(m.get('color', 0) is None for m in metas),
      'a removed colour travels as an explicit null, not as silence')
check(any(m.get('contacts', 0) is None for m in metas),
      'removed contacts travel as an explicit null')
print()
print(f'{len(viewer_html) // 1024} KB in the viewer cell,'
      f' {sum(len(u) for u in updates) // 1024} KB in {len(updates)} update cells')
print('=== 5a ALL PASS' if not bad else f'=== 5a {len(bad)} FAILED')


### 5b · The channel itself, in this browser

This asks Colab directly. It opens the same `BroadcastChannel` the viewer uses,
posts `viewerReady` on it, and counts the update cells that answer.

A non-zero count proves three things at once: the channel **crosses Colab's
output iframes**, the update cells from section 1 are **still listening**, and
they **replay on demand** — which is exactly what makes section 4 work.

*(Re-run section 1's `add()` cells first if you have restarted the runtime.)*


In [ ]:
from google.colab import output as _out

vid = v.config['viewer_id']
js = '''
(async () => {
  const heard = [];
  const ch = new BroadcastChannel('py2dmol_' + %r);
  ch.onmessage = (e) => {
    const d = e.data || {};
    if (d.operation && d.operation !== 'viewerReady') heard.push(d.seq);
  };
  ch.postMessage({operation: 'viewerReady', sourceInstanceId: 'probe'});
  await new Promise(r => setTimeout(r, 1200));
  ch.close();
  return JSON.stringify(heard.sort((a, b) => a - b));
})()
''' % vid

try:
    raw = _out.eval_js(js, timeout_sec=20)
except Exception as e:
    raw = None
    print('eval_js failed:', e)
answered = json.loads(raw or '[]')
print(f'{len(answered)} cell outputs answered the announcement: seq {answered}')
if answered:
    print('PASS  BroadcastChannel crosses Colab output iframes, and the update'
          ' cells replay on request')
else:
    print('FAIL  nothing answered - either no update cells are live in this page,'
          ' or the channel is not crossing iframes in this Colab build')


---

If sections 1–3 looked right, 4 came back with three frames and no runtime, and
5a and 5b both passed, the Colab path is doing everything it claims.

Anything that failed is worth reporting with the section number — the local
harness (`python3 tests/colab.py`) covers the same ground and will say whether
it reproduces outside Colab.


## 6 · Could the cells share one copy of the library?

Every `show()` writes ~430 KB into its own cell output, because Colab gives
each output its own iframe and a script in one is invisible to the next. Ten
viewers in a notebook is 4.3 MB of `.ipynb`.

They are same-origin — that is why `BroadcastChannel` works — so in a plain
same-origin iframe layout a later cell can reach a sibling through `parent`,
read the library out of it, and either evaluate a copy or call the neighbour's
function directly. All four of those were measured locally and all four work.

**What is not known is whether real Colab allows it.** `BroadcastChannel` only
needs a shared origin; reaching `parent.document` and enumerating siblings is a
stronger ask, and Colab's output frames may sit under a different parent than
they appear to. This cell asks your browser.

*(Run after section 1, so there is more than one viewer output on the page.)*


In [ ]:
from google.colab import output as _out
import json

js = '''
(() => {
  const r = {parentReachable: false, siblings: 0, readable: 0, lender: false};
  try {
    const d = parent.document;          // the stronger ask
    r.parentReachable = true;
    const fr = [...d.querySelectorAll('iframe')];
    r.siblings = fr.length;
    for (const f of fr) {
      try {
        const w = f.contentWindow;
        if (!w) continue;
        r.readable++;                    // reading it at all is the test
        if (w.py2dmol_viewers && Object.keys(w.py2dmol_viewers).length) r.lender = true;
      } catch (e) { r.blocked = String(e).slice(0, 80); }
    }
  } catch (e) { r.err = String(e).slice(0, 120); }
  return JSON.stringify(r);
})()
'''
try:
    r = json.loads(_out.eval_js(js, timeout_sec=20) or '{}')
except Exception as e:
    r = {'err': str(e)}
print(json.dumps(r, indent=1))
print()
if r.get('parentReachable') and r.get('readable', 0) > 1:
    print('PASS  cells can reach each other\'s frames - sharing one copy of the'
          ' library is possible here, and would take a ~430 KB payload down to'
          ' one per notebook rather than one per show()')
elif r.get('parentReachable'):
    print('PARTIAL  parent is reachable but only', r.get('readable'),
          'frame(s) could be read - see "blocked" above')
else:
    print('FAIL  the parent document is not reachable, so each cell must keep'
          ' its own copy. BroadcastChannel still works - it needs only a shared'
          ' origin - which is why the live path does.')


### 6b · …then over the channel instead

Section 6 came back **FAIL** in real Colab, with a `SecurityError`: the output
frames are same-origin *with each other* — which is why `BroadcastChannel`
works — but cross-origin with the notebook page, and `parent` is the only
route to a sibling's DOM.

So the DOM is out. The channel is not. These two cells lend and borrow a
429 KB string over it, which is the exact size of the library, without either
one touching `parent`. Locally it arrives in 25 ms.

**Run the lender first, then the borrower.**


In [ ]:
# THE LENDER. In a real implementation this is any cell that already carries
# the library; here it is a string of the same size.
from IPython.display import HTML, display
display(HTML('''
<script>
const BIG = 'x'.repeat(440215);
const ch = new BroadcastChannel('py2dmol_lib_probe');
ch.onmessage = (e) => {
  if (e.data && e.data.op === 'need') ch.postMessage({op: 'lib', src: BIG});
};
document.currentScript.parentElement.append('lender ready (440215 bytes)');
</script>'''))


In [ ]:
# THE BORROWER, in its own output frame - so its own window, its own globals,
# and no access to the lender's DOM.
from IPython.display import HTML, display
display(HTML('''
<div id="borrow-out">asking…</div>
<script>
(async () => {
  const el = document.getElementById('borrow-out');
  let viaParent = 'blocked';
  try { void parent.document; viaParent = 'reachable'; } catch (e) {}
  const t0 = performance.now();
  const ch = new BroadcastChannel('py2dmol_lib_probe');
  const src = await new Promise((res) => {
    const t = setTimeout(() => res(null), 8000);
    ch.onmessage = (e) => { if (e.data && e.data.op === 'lib') { clearTimeout(t); res(e.data.src); } };
    ch.postMessage({op: 'need'});
  });
  const ms = Math.round(performance.now() - t0);
  el.textContent = src
    ? ('PASS  borrowed ' + src.length + ' bytes in ' + ms + ' ms'
       + '  (parent.document is ' + viaParent + ', and was not used)')
    : ('FAIL  nothing answered in 8s - run the lender cell above first;'
       + ' if it did run, the channel does not carry a payload this size here');
})();
</script>'''))


### 6c · Does Python get a straight answer?

`` has to decide, at `show()` time, whether anything on the
page can lend. A module flag records what this **kernel** has written, which is
a guess at what the **page** still has — clear the lending cell's output and the
guess is wrong in the direction that breaks every later viewer.

Colab can be asked: `eval_js` runs in the output frame and returns a value to
the kernel. Jupyter has no synchronous equivalent, so there the flag stands.
This checks the ask works here, and that it changes its mind when it should.


In [ ]:
from py2Dmol.viewer import _lender_on_page
import py2Dmol.viewer as _vm

KEY = 'bundles/py2Dmol.notebook.min.js'
before = _lender_on_page(KEY)
print('before any sharing viewer :', before)

_vm._LENT_BUNDLE = None          # pretend a fresh kernel
w = py2Dmol.view((320, 320))
w.add(helix(0), align=False, name='lender')
w.show()


In [ ]:
# ...and again, now that a lender is on the page.
after = _lender_on_page(KEY)
print('with a lender on the page :', after)
print()
if before is None:
    print('SKIP  eval_js is unavailable here, so Python cannot ask and falls'
          ' back to the kernel flag - which is what Jupyter always does')
elif before is False and after is True:
    print('PASS  the ask works: Python inlines when nothing can lend and'
          ' borrows when something can, rather than guessing from what this'
          ' kernel happens to have written')
elif before and after:
    print('NOTE  something was already lending before this cell - re-run from a'
          ' fresh runtime to see the transition')
else:
    print(f'FAIL  before={before} after={after} - the page did not answer once'
          ' a lender was on it')
